# Module 12 — Exception Handling

**Course:** Programming Fundamentals Using Python

**Navigation:** [01 Introduction](01_Introduction_to_Programming_and_Python.ipynb) | [02 Variables](02_Variables_Data_Types_and_Operators.ipynb) | [03 Input/Output](03_Input_and_Output.ipynb) | [04 Strings](04_Strings.ipynb) | [05 Decisions](05_Decision_Making.ipynb) | [06 Loops](06_Loops_and_Iteration.ipynb) | [06b Mini Project](06b_Mini_Project_Quiz_and_Grade_Report.ipynb) | [07 Lists](07_Lists.ipynb) | [08 Tuples/Sets/Dicts](08_Tuples_Sets_and_Dictionaries.ipynb) | [09 Functions](09_Functions.ipynb) | [10 Modules](10_Modules_and_Packages.ipynb) | [11 Files](11_File_Handling.ipynb) | **[12 Exceptions](12_Exception_Handling.ipynb)** | [13 Problem Solving](13_Problem_Solving_and_Algorithmic_Thinking.ipynb) | [14 OOP](14_Introduction_to_OOP.ipynb) | [15 Data Handling](15_Python_Data_Handling.ipynb) | [16 Computational Thinking](16_Computational_Thinking.ipynb)

---

## Module overview

Every program meets the unexpected: the user types "twenty" instead of 20,
a file is missing, a list is shorter than assumed. **Exceptions** are
Python's way of raising an alarm; **exception handling** is how your
program answers the alarm gracefully instead of crashing.

## Learning objectives

By the end of this module you can:

* distinguish syntax, runtime and logical errors — and exceptions
* recognise the everyday exceptions: `ValueError`, `TypeError`,
  `ZeroDivisionError`, `IndexError`, `KeyError`, `FileNotFoundError`,
  `NameError`
* write `try` / `except` / `else` / `finally` blocks correctly
* catch specific exceptions (and explain why bare `except` is avoided)
* raise exceptions with `raise` for invalid data
* turn a fragile interactive program into a robust one

## Prerequisites

Modules 1–11; Module 11's `FileNotFoundError` preview is extended here.

## Contents

1. Errors vs exceptions — the taxonomy
2. A field guide to common exceptions
3. `try` / `except`
4. Catching specific exceptions — and why bare `except` is avoided
5. Multiple exception types
6. `else` — the no-crash path
7. `finally` — always runs
8. `raise` — throwing your own alarms
9. Input validation patterns
10. Mini project: Robust Interactive Application
11. Common mistakes
12. Lab 12 — Exceptions in practice
13. Exercises
14. Summary and final assessment

## 1. Errors vs exceptions — the taxonomy

### Concept

| Kind | When noticed | Example | Can handling help? |
|---|---|---|---|
| **Syntax error** | before running (Python can't even read the code) | `if x = 5:` | No — fix the code |
| **Runtime error / exception** | while running | `int("abc")` | **Yes — this module** |
| **Logical error** | never crashes, but the answer is wrong | average of wrong column | No — testing & tracing (Module 13) |

An **exception** is an object Python creates when something unexpected
happens. Uncaught, it stops the program and prints a traceback. Caught, it
becomes information your program can act on.

### Example — three failures, three lessons

In [ ]:
# 1) Syntax error — commented out, because Python refuses to run the cell:
# if 5 > 3      # SyntaxError: expected ':'

# 2) Runtime error (exception):
numbers = [1, 2, 3]
# print(numbers[10])      # IndexError — would crash the cell

# 3) Logical error — runs fine, wrong answer:
average = 70 + 80 + 90 / 3          # only 90 is divided!
print("Wrong average:", average)    # 180.0, not 80

Syntax errors stop everything before the first line runs. Exceptions
crash *at the line* where they occur. Logical errors are the sneakiest —
only testing and tracing find them (Module 13).

## 2. A field guide to common exceptions

### Concept

| Exception | Raised when... | Typical trigger |
|---|---|---|
| `ValueError` | right type, impossible value | `int("abc")` |
| `TypeError` | wrong type for the operation | `"age: " + 20` |
| `ZeroDivisionError` | dividing by zero | `10 / 0` |
| `IndexError` | index outside the sequence | `items[10]` on 3 items |
| `KeyError` | dictionary key missing | `d["missing"]` |
| `FileNotFoundError` | opening a missing file | `open("nope.txt")` |
| `NameError` | name never defined | typo'd variable |

### Example — each one, live (caught so the notebook survives!)

In [ ]:
cases = [
    ("int('abc')", "ValueError"),
    ("'age: ' + 20", "TypeError"),
    ("10 / 0", "ZeroDivisionError"),
    ("[1, 2, 3][10]", "IndexError"),
    ("{'a': 1}['b']", "KeyError"),
    ("open('no_such_file.txt')", "FileNotFoundError"),
    ("undefined_variable", "NameError"),
]

for label, kind in cases:
    print(f"{kind:20} <- {label}")

(Each row names what Python *would* raise. Try one in a scratch cell and
compare its traceback with the table — reading tracebacks is a skill
Module 13 drills.)

## 3. `try` / `except`

### Concept

Wrap risky statements in `try:`; if an exception occurs there, Python
jumps to the matching `except:` block instead of crashing. Execution
continues *after* the try statement.

### Syntax

```python
try:
    risky_statement
except SomeException:
    recovery_plan
```

### Example 1 — Beginner: safe conversion

In [ ]:
text = "abc"                 # imagine this came from input()

try:
    number = int(text)
    print("Converted:", number)
except ValueError:
    print("That was not a whole number.")

print("Program continues normally.")

### Explanation

* `int("abc")` raises `ValueError` — but inside `try`, the alarm is caught.
* The `except ValueError:` block runs; the crash never happens.
* The line *after* the try statement runs either way — the program lives.

### Example 2 — Practical: zero-division with a message

In [ ]:
total_marks = 0
subject_count = 0

try:
    average = total_marks / subject_count
    print(f"Average: {average:.1f}")
except ZeroDivisionError:
    print("Cannot compute an average of zero subjects.")

(An `if subject_count == 0:` check would also work — Module 16 compares
"look before you leap" with "ask forgiveness". Exceptions shine when the
check is hard, racy, or impossible.)

## 4. Catching specific exceptions — why bare `except` is avoided

### Concept

A bare `except:` catches **every** exception — including `KeyboardInterrupt`
(Ctrl+C!) and, worse, the `TypeError` that reveals a genuine bug in your
logic. Catching it silently *hides the bug*. Catch only what you expect
and can meaningfully handle.

In [ ]:
value = "12x"

try:
    number = int(value)
except ValueError:                          # exactly the expected problem
    print(f"Could not convert {value!r} to an integer.")

```python
# The anti-pattern:
# try:
#     process_data(data)
# except:                       # catches EVERYTHING
#     pass                      # ...including your bugs. Never do this.
```

## 5. Multiple exception types

### Concept

One `try` can have several `except` clauses; the first matching one wins.
Group sibling exceptions in a tuple when the recovery is the same.

In [ ]:
inputs = [("42", "int"), ("3.9", "int"), ("7", "int")]

for text, kind in inputs:
    try:
        if kind == "int":
            number = int(text)
        else:
            number = float(text)
        print(f"{text} -> {number} ({type(number).__name__})")
    except ValueError:
        print(f"{text} is not a valid {kind}")

In [ ]:
def safe_get(items, index):
    try:
        return items[index]
    except IndexError:
        return None                  # out of range -> no value
    except TypeError:                # e.g. items was not a list at all
        return None


print(safe_get([1, 2, 3], 1))
print(safe_get([1, 2, 3], 99))
print(safe_get("abc", 1))       # strings ARE indexable -> 'b' comes back

## 6. `else` — the no-crash path

### Concept

`else:` runs **only if no exception occurred** in the `try` block. It
keeps success-path code out of the `try` (so its bugs can't be silently
caught) while still benefiting from the shared flow.

In [ ]:
text = "150"

try:
    marks = int(text)
except ValueError:
    print("Invalid number.")
else:
    # runs ONLY when int() succeeded:
    print(f"Valid marks: {marks}")
    print("Twice:", marks * 2)

In [ ]:
text = "1o0"                       # note the letter O

try:
    marks = int(text)
except ValueError:
    print("Invalid number.")
else:
    print(f"Valid marks: {marks}")

## 7. `finally` — always runs

### Concept

`finally:` runs **no matter what** — success, handled error, even an
unhandled one (before the crash). Use it for clean-up that must happen:
closing connections, releasing locks, printing "attempt finished".

In [ ]:
attempts_log = []

def parse(text):
    try:
        value = int(text)
    except ValueError:
        attempts_log.append(f"failed on {text!r}")
        return None
    else:
        attempts_log.append(f"parsed {value}")
        return value
    finally:
        attempts_log.append("attempt finished")   # always appended


print(parse("42"))
print(parse("oops"))
print(attempts_log)

(The nested quotes in the list's repr are Python's own printout — proof
that the log recorded both attempts, each finished by `finally`.)

The classic real use is files (Module 11): `with open(...)` is preferred
because it does the `finally: file.close()` dance for you.

## 8. `raise` — throwing your own alarms

### Concept

Your functions can signal invalid data with `raise ExceptionType("message")`.
The caller then decides how to handle it — the same machinery Python uses.

In [ ]:
def set_age(age):
    if not isinstance(age, int):
        raise TypeError("age must be an int")
    if age < 0 or age > 130:
        raise ValueError("age must be between 0 and 130")
    return age


try:
    set_age(-5)
except ValueError as error:          # 'as error' names the exception object
    print("Rejected:", error)

try:
    set_age(30)
except (TypeError, ValueError):
    print("Not reached")
else:
    print("Age 30 accepted.")

### Example — Practical: validating exam marks

In [ ]:
def validate_marks(marks):
    """Accept marks 0..100; raise ValueError otherwise."""
    if marks < 0 or marks > 100:
        raise ValueError(f"marks must be 0-100, got {marks}")
    return marks


for candidate in (85, -3, 104):
    try:
        validate_marks(candidate)
        print(f"{candidate}: accepted")
    except ValueError as error:
        print(f"{candidate}: {error}")

## 9. Input validation patterns

### Concept

Interactive programs ask users for numbers — and users obligingly type
anything. The workhorse pattern: **loop until valid**, converting inside
`try`:

```python
while True:
    try:
        value = int(input("Whole number: "))
        break              # success: leave the loop
    except ValueError:
        print("Please type a whole number.")
```

(Here we *simulate* the typing so the notebook runs unattended.)

In [ ]:
def read_whole_number(simulated):
    """Keep asking until simulated typing yields a whole number."""
    for attempt in simulated:            # in a real program: while True + input()
        try:
            value = int(attempt)
            return value
        except ValueError:
            print(f"{attempt!r} is not a whole number - try again.")
    return None


print(read_whole_number(["abc", "12.5", "42"]))

In [ ]:
# Validating a menu choice and a positive number:
def read_choice(simulated, valid):
    for attempt in simulated:
        cleaned = attempt.strip().lower()
        if cleaned in valid:
            return cleaned
        print(f"Choose one of: {', '.join(sorted(valid))}")
    return None


print(read_choice(["water", " TEA "], {"tea", "coffee"}))

In [ ]:
def read_positive(simulated):
    for attempt in simulated:
        try:
            value = float(attempt)
        except ValueError:
            print(f"{attempt!r} is not a number.")
            continue
        if value > 0:
            return value
        print("It must be positive.")


print(read_positive(["-5", "zero", "3.5"]))

Hmm — that last run needs a careful eye: `"-5"` converts to a *number*
fine, so it's the positivity check (not ValueError) that complains;
`"zero"` is the one that isn't a number. The output shows both rejection
reasons in the order the attempts arrived. Tracing that is exactly the
Module 13 skill.

## 10. Mini project — Robust Interactive Application

**Requirements**

1. A marks-entry program: menu of add / show / average / quit.
2. Every numeric read is guarded; invalid input never crashes.
3. Marks outside 0–100 are rejected with a clear message (raise + except).
4. Average handles the empty case with an explicit message.

**Pseudocode**

```text
marks <- []
LOOP:
    choice <- menu
    add:    raw <- ask
            TRY n <- int(raw) EXCEPT: message; continue
            TRY validate(n) (0..100) EXCEPT ValueError as e: print e; continue
            marks.append(n)
    show:   IF empty: message ELSE print list
    average: IF empty: message ELSE print mean
    quit:   break
```

In [ ]:
# ----- Robust Marks Entry -----
def validate_marks(value):
    """Raise ValueError unless value is an integer 0..100."""
    if value < 0 or value > 100:
        raise ValueError(f"marks must be 0-100, got {value}")
    return value


marks = []

while True:
    print()
    print("--- Marks Entry ---")
    print("add / show / average / quit")
    choice = input("Choice: ").strip().lower()

    if choice == "add":
        raw = input("Marks: ").strip()
        try:
            value = int(raw)                 # may raise ValueError
        except ValueError:
            print(f"{raw!r} is not a whole number.")
            continue
        try:
            validate_marks(value)            # may raise ValueError
        except ValueError as error:
            print("Rejected:", error)
            continue
        marks.append(value)
        print(f"Recorded {value}.")
    elif choice == "show":
        if len(marks) == 0:
            print("(no marks yet)")
        else:
            for position, score in enumerate(marks, start=1):
                print(f"{position}. {score}")
    elif choice == "average":
        if len(marks) == 0:
            print("No marks to average.")
        else:
            print(f"Average: {sum(marks) / len(marks):.1f}")
    elif choice == "quit":
        print("Goodbye!")
        break
    else:
        print("Unknown choice.")

### Sample run (the session used to validate this notebook)

```text
add 95   -> Recorded 95.
add 120  -> Rejected: marks must be 0-100, got 120
add abc  -> 'abc' is not a whole number.
add 80   -> Recorded 80.
show     -> 1. 95 / 2. 80
average  -> Average: 87.5
quit     -> Goodbye!
```

**Bad vs improved:** the original Module 3 calculator crashed on the first
typo; this version *never* crashes and explains every rejection. Note what
exceptions did NOT do: they did not hide the "120" case — that was
deliberate validation with `raise`.

## 11. Common mistakes — the module in one list

```python
# 1. Bare except: pass          -> hides real bugs (and Ctrl+C!)
# 2. except Exception around huge blocks -> errors surface far from cause
# 3. Wrong exception type caught -> e.g. except TypeError for int("x")
# 4. Using exceptions to silence bugs (IndexError on [0] instead of
#    checking emptiness) — validate the *known* way, catch the *rest*.
# 5. Forgetting else/finally ordering: try -> except... -> else -> finally.
# 6. raise ValueError without a message — future-you deserves context.
```

## 12. Lab 12 — Exceptions in practice

### Learning objectives

Convert earlier fragile programs into robust ones; practise catching the
*right* exception; use `raise` for domain rules.

### Short theory recap

`try` wraps risk; `except E:` handles expected alarms (be specific);
`else:` runs on success; `finally:` always runs; `raise` signals invalid
data. Bare `except` hides bugs — avoid.

### Warm-up (run and predict)

1. `try: int("7.5")` / `except ValueError: print("no")` — which runs?
2. Same but with `except TypeError:` — what happens to the program?
3. `try: print("a") finally: print("b")` — order of letters?

### Guided exercises

In [ ]:
# Guided 1 — harden Module 9's safe_divide properly.
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None


print(safe_divide(10, 4))
print(safe_divide(10, 0))

In [ ]:
# Guided 2 — catch two different exceptions, two different messages.
def parse_int(text):
    try:
        return int(text)
    except ValueError:
        print(f"ValueError: {text!r} is not an integer.")
    except TypeError:
        print(f"TypeError: expected text, got {type(text).__name__}.")


print(parse_int("55"))
print(parse_int("5.5"))
print(parse_int(None))        # int(None) raises TypeError

In [ ]:
# Guided 3 — a file read that survives a missing file (Module 11 payoff).
from pathlib import Path

def read_config(path):
    try:
        return path.read_text(encoding="utf-8")
    except FileNotFoundError:
        return ""


config = Path("file_sandbox") / "config.txt"
config.parent.mkdir(exist_ok=True)
print(len(read_config(config)), "characters before creating")
config.write_text("mode=fast\n", encoding="utf-8")
print(len(read_config(config)), "characters after creating")

### Independent exercises

1. Wrap `int("3.9")` in try/except and print the error's *message* using
   `as error`.
2. Write `safe_index(items, i)` returning `None` on `IndexError`.
3. Write `safe_lookup(d, key)` returning a default on `KeyError` — then
   compare with Module 8's `.get()`.
4. Write a `finally` demo that prints "done" whether or not division
   succeeded.
5. Write `read_age(text)` raising `ValueError` for non-ints and for ages
   outside 5–100; catch and print both cases.
6. Convert Module 6's number-reading loop to the try-until-valid pattern
   (simulated inputs).
7. Catch `ZeroDivisionError` around a *list* average that divides by
   `len(marks)`; message on empty.
8. Write `open_or_default(path)` returning the file's text or "" — test
   on existing and missing files.
9. Group `except (TypeError, ValueError)` for a converter handling both
   "abc" and `None` inputs.
10. Re-run a previous module's snippet that crashed; wrap it so it now
    prints a friendly message instead.

### Challenge problems

1. **Robust CSV loader.** Read a CSV (Module 11 style) where rows may have
    bad numbers; skip bad rows with a warning, count skipped, return good
    rows.
2. **Retry budget.** A validation loop that gives up after 3 failed
    attempts, raising `ValueError("too many attempts")`, caught by the
    caller with a friendly message.
3. **Exception zoo.** One data list containing inputs for all seven
    exceptions from section 2; a generic `attempt(call_description)`
    driver that catches *named* exceptions per item and prints a table of
    outcomes.

## 13. Exercises

## Quick Review

1. Difference between a syntax error and an exception?
2. Which exception does `int("abc")` raise? And `"a" + 1`?
3. What does `try/except` prevent?
4. Why avoid bare `except:`?
5. When does `else` run? When does `finally` run?
6. What does `except ValueError as error:` bind?
7. How do you raise a ValueError with a message?
8. Can one `try` have multiple `except` blocks? Which wins?
9. Give a case where a check (`if`) is better than try/except.
10. Why is `except Exception: pass` dangerous?

<details><summary>Answers</summary>

1. Syntax errors stop the program before it runs; exceptions occur while
   it runs and can be caught.
2. `ValueError`; `TypeError`.
3. The crash (and the traceback) — the program continues.
4. It catches everything, including real bugs and Ctrl+C, hiding problems.
5. `else` only when no exception occurred; `finally` always.
6. The exception *object*, e.g. printing `error` shows the message.
7. `raise ValueError("message here")`.
8. Yes — the first matching clause wins.
9. When the check is simple and local, e.g. `if items:` before dividing.
10. It swallows bugs silently, making them undiagnosable.

</details>

## Predict the Output

In [ ]:
# Snippet 1 — prediction:
try:
    print(int("10"))
except ValueError:
    print("bad")
print("end")

In [ ]:
# Snippet 2 — prediction:
try:
    print(10 / 0)
except ZeroDivisionError:
    print("caught")
finally:
    print("finally")

In [ ]:
# Snippet 3 — prediction:
try:
    x = int("5")
except ValueError:
    x = 0
else:
    x = x * 2
print(x)

In [ ]:
# Snippet 4 — prediction:
items = [1, 2]
try:
    print(items[5])
except IndexError:
    print("IndexError")
except KeyError:
    print("KeyError")

In [ ]:
# Snippet 5 — prediction:
def f(n):
    if n < 0:
        raise ValueError("negative!")
    return n * 2


try:
    print(f(-1))
except ValueError as e:
    print(e)

In [ ]:
# Snippet 6 — prediction:
log = []
try:
    log.append("try")
    int("x")
except ValueError:
    log.append("except")
finally:
    log.append("finally")
print(log)

<details><summary>Answers</summary>

1. `10` then `end` (no exception).
2. `caught` then `finally` — both always run in that order.
3. `10` — else doubled the parsed value.
4. `IndexError` — first matching except clause.
5. `negative!` — the raise carried its message.
6. `['try', 'except', 'finally']` — full lifecycle in order.

</details>

## Find the Error

**Error 1**

```python
try:
    value = int("abc")
print(value)
except ValueError:
    print("bad")
```

**Error 2**

```python
try:
    total = 10 / 0
except:
    pass
```

**Error 3**

```python
try:
    number = int(input_text)
except ValueError
    number = 0
```

**Error 4**

```python
try:
    marks = int("90")
else:
    print(marks)
```

**Error 5**

```python
try:
    raise "oops"
except ValueError:
    print("caught")
```

**Error 6**

```python
try:
    data = open("missing.txt").read()
except FileNotFoundError:
    pass
print(data)
```

<details><summary>Answers</summary>

1. Statements can't sit between `try` and `except`; also `value` would be
   undefined. Move the print into `else`.
2. It runs, but the bare except + pass hides *why* it failed — catch
   `ZeroDivisionError` and act.
3. Missing colon after `except ValueError`.
4. `else` requires a preceding `except` clause (try/else alone is a
   `SyntaxError`).
5. You can only raise exception *instances*: `raise ValueError("oops")`.
6. If the open failed, `data` was never assigned → later `NameError`.
   Return/handle inside the except block.

</details>

## Programming Exercises

1. **Safe int.** `parse_int(text)` returning the number or `None`; demo on
   "42", "4.2", "abc".
2. **Safe float with retry.** Loop (simulated) until a valid float appears;
   count the attempts.
3. **Grade guard.** `letter_grade(marks)` that *raises* ValueError below 0
   or above 100; catch and print for 5 test values.
4. **Menu validator.** `read_choice` from section 9, but case-insensitive
   and strip-happy; test with messy inputs.
5. **List average.** try/except around empty-list division; message "no
   data yet".
6. **Dictionary lookup.** Two versions — try/except KeyError and `.get()`;
   print both results for a missing key.
7. **finally counter.** A function that logs "called" in `finally` and
   returns success/failure; call 3 times, print the log.
8. **File shield.** `count_lines(path)` returning the count or -1 on
   FileNotFoundError; test both paths.
9. **Type-mixed adder.** `add(a, b)` raising TypeError for non-numbers;
   caller catches and reports.
10. **Crash-to-message conversion.** Take three old crashing snippets from
    earlier notebooks (int conversion, index, division) and wrap each.

## Challenge Problems

1. **Input firewall.** A single function `read_value(simulated, kind,
    low, high)` converting to int or float, validating the range with
    `raise`, retrying with messages; a table demo of 6 tricky inputs.
2. **Transaction log.** Wrap operations in try/except/else/finally writing
    a log line each ("started/ok|failed: msg/finished") to a sandbox file;
    print the log at the end.
3. **Robust quiz.** Rebuild Module 6b's quiz answer-reading so that any
    input is safe (strip/upper, ValueError-proof scoring), plus a custom
    `InvalidAnswerError` raised after 3 bad answers.

## 14. Summary

* Syntax errors stop the program before it starts; exceptions occur
  during it; logical errors only testing finds.
* Know the big seven by name and trigger: ValueError, TypeError,
  ZeroDivisionError, IndexError, KeyError, FileNotFoundError, NameError.
* `try` wraps risk; specific `except` clauses handle it; `else` celebrates
  success; `finally` always cleans up.
* Bare `except` hides bugs — catch only what you expect.
* `raise` signals invalid data with a message; callers decide the policy.
* Validation loops convert fragile input code into polite, uncrashable
  programs.

## Final assessment (check yourself)

1. Write the four-block skeleton and comment when each block runs.
2. `except ValueError: pass` around `int(x)` — why is this better than
    bare except but still incomplete?
3. Why does the "loop until valid" pattern need `try` *inside* the loop
    rather than around it?
4. A teammate silences every exception to "stop the crashes". Argue the
    case in three sentences using two concrete exceptions.

---

**Next up:** problem solving and algorithmic thinking — how to attack
problems methodically before typing a line.

| Previous | Next |
|---|---|
| [Module 11 — File Handling](11_File_Handling.ipynb) | [Module 13 — Problem Solving and Algorithmic Thinking](13_Problem_Solving_and_Algorithmic_Thinking.ipynb) |